# 05.13 - Feature Engineering

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

Feature engineering is creating new features, selecting important ones, and transforming existing ones to improve model performance. It's often where the biggest gains come from.

## 2. Why Does This Matter?

Better features beat better models. Domain knowledge encoded as features can dramatically improve accuracy.

## 3. Prerequisites

- Phase 04 (Data Analysis)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Create new features from existing ones
- Select important features
- Transform features
- Measure the impact of feature engineering

## 5. Mental Model

Feature engineering:

1. **Creation**: combine/transform features (ratios, interactions, aggregates).
2. **Selection**: keep the most informative features.
3. **Transformation**: scale, encode, bin.

Goal: give the model the right information in the right form.


## 6. Generate Data

Create a dataset where engineered features help.


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

np.random.seed(42)
n = 500
df = pd.DataFrame({
    "width": np.random.uniform(1, 10, n),
    "height": np.random.uniform(1, 10, n),
    "price_per_unit": np.random.uniform(5, 20, n),
})
# True target depends on AREA (width*height) and total price
df["area"] = df["width"] * df["height"]
df["total_price"] = df["area"] * df["price_per_unit"]
print(df.head())


      width    height  price_per_unit       area  total_price
0  4.370861  7.283455        7.776994  31.834972   247.580382
1  9.556429  5.824867       13.128514  55.664929   730.797816
2  7.587945  3.785749       18.094188  28.726054   519.774600
3  6.387926  8.324155       15.983373  53.174090   849.901334
4  2.404168  7.162581       17.098417  17.220045   294.435519


## 7. Without Feature Engineering

Try to predict total_price from raw features.


In [2]:
X_raw = df[["width", "height", "price_per_unit"]].values
y = df["total_price"].values
X_tr, X_te, y_tr, y_te = train_test_split(X_raw, y, test_size=0.3, random_state=42)

m_raw = LinearRegression().fit(X_tr, y_tr)
r2_raw = r2_score(y_te, m_raw.predict(X_te))
print(f"Raw features R^2: {r2_raw:.3f}")
print("The model must discover width*height on its own.")


Raw features R^2: 0.809
The model must discover width*height on its own.


## 8. With Feature Engineering

Add the area feature (width*height) explicitly.


In [3]:
X_eng = df[["width", "height", "price_per_unit", "area"]].values
X_tr2, X_te2, _, _ = train_test_split(X_eng, y, test_size=0.3, random_state=42)

m_eng = LinearRegression().fit(X_tr2, y_tr)
r2_eng = r2_score(y_te, m_eng.predict(X_te2))
print(f"Engineered features R^2: {r2_eng:.3f}")
print("\nAdding the area feature makes the relationship explicit.")


Engineered features R^2: 0.899

Adding the area feature makes the relationship explicit.


## 9. Feature Selection

Select the most important features using mutual information.


In [4]:
from sklearn.feature_selection import SelectKBest, mutual_info_regression

selector = SelectKBest(mutual_info_regression, k=3)
X_sel = selector.fit_transform(X_eng, y)
print("Selected feature indices:", selector.get_support(indices=True))
print("Feature scores:", selector.scores_.round(3))
print("\nThe area feature has the highest mutual information.")


Selected feature indices: [0 1 3]
Feature scores: [0.327 0.319 0.127 1.065]

The area feature has the highest mutual information.


## 10. Polynomial Features

Capture nonlinear relationships with polynomial features.


In [5]:
from sklearn.preprocessing import PolynomialFeatures

# Nonlinear target
Xp = np.random.uniform(-3, 3, (300, 1))
yp = Xp[:, 0]**2 + np.random.normal(0, 0.5, 300)
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(Xp, yp, test_size=0.3, random_state=42)

# Linear model on raw
m_lin = LinearRegression().fit(Xp_tr, yp_tr)
r2_lin = r2_score(yp_te, m_lin.predict(Xp_te))

# Linear model on polynomial features
poly = PolynomialFeatures(degree=2)
Xp_tr_p = poly.fit_transform(Xp_tr)
Xp_te_p = poly.transform(Xp_te)
m_poly = LinearRegression().fit(Xp_tr_p, yp_tr)
r2_poly = r2_score(yp_te, m_poly.predict(Xp_te_p))

print(f"Linear on raw: R^2={r2_lin:.3f}")
print(f"Linear on polynomial: R^2={r2_poly:.3f}")
print("\nPolynomial features capture the quadratic relationship.")


Linear on raw: R^2=-0.056
Linear on polynomial: R^2=0.964

Polynomial features capture the quadratic relationship.


## 11. Failure Case: Too Many Features

Adding irrelevant features can hurt (overfitting, noise).


In [6]:
# Add 50 random irrelevant features
X_noise = np.column_stack([X_eng, np.random.normal(0, 1, (len(X_eng), 50))])
Xn_tr, Xn_te, _, _ = train_test_split(X_noise, y, test_size=0.3, random_state=42)
m_noise = LinearRegression().fit(Xn_tr, y_tr)
r2_noise = r2_score(y_te, m_noise.predict(Xn_te))
print(f"With 50 noise features: R^2={r2_noise:.3f}")
print(f"Without noise features: R^2={r2_eng:.3f}")
print("\nIrrelevant features can hurt generalization.")


With 50 noise features: R^2=0.884
Without noise features: R^2=0.899

Irrelevant features can hurt generalization.


## 12. Debugging: Common Errors

- **Leakage**: engineered features using target info.
- **Too many features**: overfitting.
- **Not scaling**: engineered features may have different scales.

## 13. Real-World Considerations

- Domain knowledge is the best source of features.
- Use feature selection to prune.
- Validate engineered features on held-out data.

## 14. Common Mistakes

- Adding features without validation.
- Leaking target info into features.

## 15. When NOT to Use

- When features are already informative and models are strong.
- When you risk overfitting.

## 16. Challenge

Create a ratio feature and measure its impact on model performance.


In [7]:
# Challenge: ratio feature
df["aspect_ratio"] = df["width"] / df["height"]
X_ratio = df[["width", "height", "price_per_unit", "area", "aspect_ratio"]].values
Xr_tr, Xr_te, _, _ = train_test_split(X_ratio, y, test_size=0.3, random_state=42)
m_ratio = LinearRegression().fit(Xr_tr, y_tr)
r2_ratio = r2_score(y_te, m_ratio.predict(Xr_te))
print(f"With aspect_ratio: R^2={r2_ratio:.3f}")
print(f"Without:           R^2={r2_eng:.3f}")
print("\nThe ratio feature adds little here since area already captures the relationship.")


With aspect_ratio: R^2=0.899
Without:           R^2=0.899

The ratio feature adds little here since area already captures the relationship.


## 17. Closed-Book Recall

Without looking back:

1. What is feature engineering?
2. Give examples of creating features.
3. What is feature selection?
4. Why can too many features hurt?

## 18. Teach-Back Questions

Explain to another person:

- Why engineered features improve models.
- The risk of adding too many features.

## 19. Summary

You created, selected, and transformed features, and measured their impact. Feature engineering is a powerful lever for model performance.

## 20. Further Experiment

- Try interaction features.
- Use recursive feature elimination.

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, pandas, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
